# BiGRU Attention for Pain Classification

Notebook with:
- Time-series preprocessing with windowing
- BiGRU + Attention model for labels prediction
- K-fold ensemble training

In [3]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score

import optuna

# ---- Config ----
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Using device:", DEVICE)

Using device: cuda


## Load data

In [4]:
TRAIN_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train.csv"
TRAIN_LABELS_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train_labels.csv"
TEST_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
train_labels_df = pd.read_csv(TRAIN_LABELS_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape (features):", train_df.shape)
print("Train labels shape:", train_labels_df.shape)
print("Test shape:", test_df.shape)

ID_COL = "sample_index"
TIME_COL = "time"
TARGET_COL = "label"
STATIC_COL = "is_pirate"

train_df = train_df.merge(train_labels_df[[ID_COL, TARGET_COL]], on=ID_COL, how="left")

print("\nTrain columns:", train_df.columns.tolist())
print("Example rows:")
display(train_df.head())

Train shape (features): (105760, 40)
Train labels shape: (661, 2)
Test shape: (211840, 40)

Train columns: ['sample_index', 'time', 'pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'n_legs', 'n_hands', 'n_eyes', 'joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29', 'joint_30', 'label']
Example rows:


,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,n_legs,n_hands,n_eyes,joint_00,...,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,joint_30,label
0,0,0,2,0,2,1,two,two,two,1.094705,...,1.945042e-06,0.000004,1.153299e-05,0.000004,0.017592,0.013508,0.026798,0.027815,0.5,no_pain
1,0,1,2,2,2,2,two,two,two,1.135183,...,6.765107e-07,0.000006,4.643774e-08,0.000000,0.013352,0.000000,0.013377,0.013716,0.5,no_pain
2,0,2,2,0,2,2,two,two,two,1.080745,...,1.698525e-07,0.000001,2.424536e-06,0.000003,0.016225,0.008110,0.024097,0.023105,0.5,no_pain
3,0,3,2,2,2,2,two,two,two,0.938017,...,5.511079e-07,0.000002,5.432416e-08,0.000000,0.011832,0.007450,0.028613,0.024648,0.5,no_pain
4,0,4,2,2,2,2,two,two,two,1.090185,...,1.735459e-07,0.000002,5.825366e-08,0.000007,0.005360,0.002532,0.033026,0.025328,0.5,no_pain


## Build dynamic & static features and scale

- Dynamic features: all time-varying numeric columns except id, time, and label.
- Static features: `is_pirate`.
- We scale only dynamic features with `StandardScaler`.

In [5]:
def check_pirate(row):
    return 0 if row.get("n_legs", "two") == "two" else 1

if "n_legs" in train_df.columns:
    train_df["is_pirate"] = train_df.apply(check_pirate, axis=1)
    test_df["is_pirate"]  = test_df.apply(check_pirate, axis=1)

    for col in ["n_legs", "n_hands", "n_eyes"]:
        if col in train_df.columns:
            train_df.drop(columns=col, inplace=True)
        if col in test_df.columns:
            test_df.drop(columns=col, inplace=True)

if "joint_30" in train_df.columns:
    train_df.drop(columns="joint_30", inplace=True)
if "joint_30" in test_df.columns:
    test_df.drop(columns="joint_30", inplace=True)

train_df.head()

,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,joint_00,joint_01,joint_02,joint_03,...,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,label,is_pirate
0,0,0,2,0,2,1,1.094705,0.985281,1.018302,1.010385,...,1.945042e-06,0.000004,1.153299e-05,0.000004,0.017592,0.013508,0.026798,0.027815,no_pain,0
1,0,1,2,2,2,2,1.135183,1.021175,0.994343,1.052364,...,6.765107e-07,0.000006,4.643774e-08,0.000000,0.013352,0.000000,0.013377,0.013716,no_pain,0
2,0,2,2,0,2,2,1.080745,0.962842,1.009588,0.977169,...,1.698525e-07,0.000001,2.424536e-06,0.000003,0.016225,0.008110,0.024097,0.023105,no_pain,0
3,0,3,2,2,2,2,0.938017,1.081592,0.998021,0.987283,...,5.511079e-07,0.000002,5.432416e-08,0.000000,0.011832,0.007450,0.028613,0.024648,no_pain,0
4,0,4,2,2,2,2,1.090185,1.032145,1.008710,0.963658,...,1.735459e-07,0.000002,5.825366e-08,0.000007,0.005360,0.002532,0.033026,0.025328,no_pain,0


In [6]:
def add_cyclical_features(df, time_col='time'):
    """
    Adds cyclical time features:
    - Periodicity of 5 and 4, deduced from Features
    """
    # Period 5 encoding
    df['sin_5'] = np.sin(2 * np.pi * (df[time_col] % 5) / 5).astype(np.float32)
    df['cos_5'] = np.cos(2 * np.pi * (df[time_col] % 5) / 5).astype(np.float32)
    
    # Period 4 encoding
    df['sin_4'] = np.sin(2 * np.pi * (df[time_col] % 4) / 4).astype(np.float32)
    df['cos_4'] = np.cos(2 * np.pi * (df[time_col] % 4) / 4).astype(np.float32)
    
    return df

train_df = add_cyclical_features(train_df)
test_df = add_cyclical_features(test_df)

In [7]:
# Identify feature columns
all_feature_cols = [c for c in train_df.columns if c not in [ID_COL, TIME_COL, TARGET_COL]]
test_feature_cols = [c for c in test_df.columns if c not in [ID_COL, TIME_COL]]

print("Num total features train:", len(all_feature_cols))
print("Num total features test:", len(test_feature_cols))

# Static features
STATIC_COLS = [STATIC_COL]

dyn_feature_cols = [c for c in all_feature_cols if c not in STATIC_COLS]
dyn_feature_cols_test = [c for c in test_feature_cols if c not in STATIC_COLS]

print("Num dynamic features train:", len(dyn_feature_cols))
print("Num dynamic features test:", len(dyn_feature_cols_test))
print("Dynamic feature columns:", dyn_feature_cols)

# Sort by id and time
train_df = train_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
test_df = test_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

# Scale dynamic features
scaler = StandardScaler()
train_dyn_vals = scaler.fit_transform(train_df[dyn_feature_cols])
test_dyn_vals = scaler.transform(test_df[dyn_feature_cols])

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()
train_df_scaled[dyn_feature_cols] = train_dyn_vals
test_df_scaled[dyn_feature_cols] = test_dyn_vals

# Encode labels
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(train_df_scaled[TARGET_COL].values)
print("Classes:", list(label_encoder.classes_))

# Build sequences per sample_index
def build_sequences(df_scaled, is_train=True):
    ids = df_scaled[ID_COL].unique()
    ids_sorted = np.sort(ids)
    seq_dyn = []
    seq_static = []
    seq_labels = []
    seq_ids = []

    for sid in ids_sorted:
        sub = df_scaled[df_scaled[ID_COL] == sid].sort_values(TIME_COL)
        dyn = sub[dyn_feature_cols].values  # (T, D_dyn)
        static_vals = sub[STATIC_COLS].iloc[0].values.astype(float)  # (D_static,)
        seq_dyn.append(dyn)
        seq_static.append(static_vals)
        seq_ids.append(sid)
        if is_train:
            # same label for whole sequence
            lbl = sub[TARGET_COL].iloc[0]
            seq_labels.append(lbl)

    seq_dyn = np.array(seq_dyn, dtype=object)
    seq_static = np.stack(seq_static)
    if is_train:
        seq_labels = np.array(label_encoder.transform(seq_labels))
        return seq_dyn, seq_static, seq_labels, np.array(seq_ids)
    else:
        return seq_dyn, seq_static, np.array(seq_ids)

train_dyn_seqs, train_static_seqs, y_seqs, train_ids = build_sequences(train_df_scaled, is_train=True)
test_dyn_seqs, test_static_seqs, test_ids = build_sequences(test_df_scaled, is_train=False)

print("Num train sequences:", len(train_dyn_seqs))
print("Num test sequences:", len(test_dyn_seqs))
print("Example sequence lengths (first 10):", [x.shape[0] for x in train_dyn_seqs[:10]])

Num total features train: 39
Num total features test: 39
Num dynamic features train: 38
Num dynamic features test: 38
Dynamic feature columns: ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29', 'sin_5', 'cos_5', 'sin_4', 'cos_4']
Classes: ['high_pain', 'low_pain', 'no_pain']
Num train sequences: 661
Num test sequences: 1324
Example sequence lengths (first 10): [160, 160, 160, 160, 160, 160, 160, 160, 160, 160]


## Windowing

We create overlapping windows for each sequence to increase the number of training samples.
Each window inherits the label of the whole sequence.

In [8]:
def create_windows_from_sequences(dyn_seqs, static_seqs, labels, ids,
                                  window_size=32, stride=8, is_train=True):
    dyn_windows = []
    static_windows = []
    y_windows = []
    id_windows = []

    for dyn, static_vec, lbl, sid in zip(dyn_seqs, static_seqs,
                                         labels if is_train else [None]*len(dyn_seqs),
                                         ids):
        T = dyn.shape[0]
        if T < window_size:
            continue
        for start in range(0, T - window_size + 1, stride):
            end = start + window_size
            dyn_windows.append(dyn[start:end])   # (window_size, D_dyn)
            static_windows.append(static_vec)    # (D_static,)
            id_windows.append(sid)
            if is_train:
                y_windows.append(lbl)

    dyn_windows = np.stack(dyn_windows).astype(np.float32)
    static_windows = np.stack(static_windows).astype(np.float32)
    id_windows = np.array(id_windows)
    if is_train:
        y_windows = np.array(y_windows)
        return dyn_windows, static_windows, y_windows, id_windows
    else:
        return dyn_windows, static_windows, id_windows

WINDOW_SIZE = 20
STRIDE = 10

X_dyn_win, X_static_win, y_win, ids_win = create_windows_from_sequences(
    train_dyn_seqs, train_static_seqs, y_seqs, train_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=True
)

X_dyn_test_win, X_static_test_win, test_ids_win = create_windows_from_sequences(
    test_dyn_seqs, test_static_seqs, labels=None, ids=test_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=False
)

print("Train windows shape (dyn):", X_dyn_win.shape)
print("Train windows shape (static):", X_static_win.shape)
print("Train windows labels shape:", y_win.shape)
print("Test windows shape (dyn):", X_dyn_test_win.shape)
print("Test windows shape (static):", X_static_test_win.shape)

Train windows shape (dyn): (9915, 20, 38)
Train windows shape (static): (9915, 1)
Train windows labels shape: (9915,)
Test windows shape (dyn): (19860, 20, 38)
Test windows shape (static): (19860, 1)


## Weighted Loss Computation

In [9]:
num_classes = len(np.unique(y_win))
class_counts = np.bincount(y_win, minlength=num_classes)  # [n0, n1, n2, ...]

print("class_counts:", class_counts)

# Effective number of samples (Cui et al., 2019)
beta = 0.999

effective_num = 1.0 - np.power(beta, class_counts)
effective_num = np.maximum(effective_num, 1e-8)

weights = (1.0 - beta) / effective_num

# Normalize so that average weight = 1
weights = weights / weights.mean()

print("class_weights (CB, beta={}):".format(beta), weights)

class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

class_counts: [ 840 1410 7665]
class_weights (CB, beta=0.999): [1.29273934 0.97203252 0.73522814]


## PyTorch Dataset and DataLoader

We build a dataset that returns `(dyn_window, static_features, label)` for training.

In [10]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, X_dyn, X_static, y=None):
        self.X_dyn = X_dyn
        self.X_static = X_static
        self.y = y

    def __len__(self):
        return len(self.X_dyn)

    def __getitem__(self, idx):
        dyn_np = np.asarray(self.X_dyn[idx], dtype=np.float32)
        stat_np = np.asarray(self.X_static[idx], dtype=np.float32)
        dyn = torch.from_numpy(dyn_np)
        stat = torch.from_numpy(stat_np)
        if self.y is not None:
            label = torch.tensor(self.y[idx], dtype=torch.long)
            return dyn, stat, label
        else:
            return dyn, stat, torch.tensor(-1, dtype=torch.long)  # dummy

def make_loader(dataset, batch_size, shuffle):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## BiGRU + Attention Model

The model processes dynamic features with a BiGRU and applies attention over time.
Static features are concatenated to the attended context vector before the final MLP.

In [11]:
class BiGRUAttentionNetImproved(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes,
                 dropout=0.3, bidirectional=True, batch_norm=True, static_dim=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Attention
        self.attn_linear = nn.Linear(self.num_directions * hidden_size, self.num_directions * hidden_size)
        self.attn_vector = nn.Linear(self.num_directions * hidden_size, 1, bias=False)

        # MLP head
        mlp_input_dim = self.num_directions * hidden_size + static_dim
        self.batch_norm = batch_norm
        if batch_norm:
            self.bn = nn.BatchNorm1d(mlp_input_dim)

        self.fc1 = nn.Linear(mlp_input_dim, mlp_input_dim // 2)
        self.fc2 = nn.Linear(mlp_input_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_dyn, x_static):
        # x_dyn: (B, T, D_dyn)
        # x_static: (B, D_static)
        h, _ = self.gru(x_dyn)  # (B, T, H*D)
        # Attention
        u = torch.tanh(self.attn_linear(h))         # (B, T, H*D)
        attn_scores = self.attn_vector(u).squeeze(-1)  # (B, T)
        attn_weights = torch.softmax(attn_scores, dim=1)  # (B, T)
        context = torch.sum(h * attn_weights.unsqueeze(-1), dim=1)  # (B, H*D)

        # Concatenate static features
        x = torch.cat([context, x_static], dim=1)  # (B, H*D + D_static)
        if self.batch_norm:
            x = self.bn(x)
        x = self.dropout(F.relu(self.fc1(x)))
        logits = self.fc2(x)
        return logits

In [12]:
class LogitSequenceDataset(Dataset):
    """Dataset that provides sequences of logits per sample."""
    def __init__(self, logit_sequences, labels=None):
        """
        Args:
            logit_sequences: List of arrays, each (num_windows_for_sample, num_classes)
            labels: Array of labels per sample
        """
        self.logit_sequences = logit_sequences
        self.labels = labels
        
    def __len__(self):
        return len(self.logit_sequences)
    
    def __getitem__(self, idx):
        logits_np = np.asarray(self.logit_sequences[idx], dtype=np.float32)
        logits = torch.from_numpy(logits_np)
        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return logits, label
        else:
            return logits, torch.tensor(-1, dtype=torch.long)


def collate_logit_sequences(batch):
    """Custom collate function to handle variable-length logit sequences."""
    sequences = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])
    
    # Pad sequences to same length
    max_len = max(seq.shape[0] for seq in sequences)
    num_classes = sequences[0].shape[1]
    
    padded = torch.zeros(len(sequences), max_len, num_classes, dtype=sequences[0].dtype)
    for i, seq in enumerate(sequences):
        padded[i, :seq.shape[0], :] = seq
    
    return padded, labels

In [13]:
def collect_logits_per_sample(model_paths, X_dyn, X_static, window_ids, params, device):
    """
    Collect ensemble logits for each window and group them by sample.
    
    Returns:
        logit_sequences: List of arrays, each (num_windows_for_sample, num_classes)
        sample_ids: Array of unique sample IDs in order
    """
    # Get predictions for all windows from ensemble
    test_ds = TimeSeriesWindowDataset(X_dyn, X_static, y=None)
    test_loader = make_loader(test_ds, batch_size=params["batch_size"] * 2, shuffle=False)
    
    all_fold_logits = []
    for path in model_paths:
        print(f"Loading model from {path}")
        model = BiGRUAttentionNetImproved(
            input_size=params["input_size"],
            hidden_size=params["hidden_size"],
            num_layers=params["num_layers"],
            num_classes=params["num_classes"],
            dropout=params["dropout_rate"],
            bidirectional=params["bidirectional"],
            batch_norm=params["batch_norm"],
            static_dim=params["static_dim"]
        ).to(device)
        
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
        logits = predict_logits(model, test_loader, device)
        all_fold_logits.append(logits)
    
    # Average logits across folds
    mean_logits = np.mean(all_fold_logits, axis=0)  # (N_windows, num_classes)
    
    # Group by sample_index
    unique_ids = np.unique(window_ids)
    logit_sequences = []
    
    for sid in unique_ids:
        mask = window_ids == sid
        sample_logits = mean_logits[mask]  # (num_windows_for_this_sample, num_classes)
        logit_sequences.append(sample_logits)
    
    return logit_sequences, unique_ids

## Training Utilities

Functions for one epoch of training/validation with optional L1/L2 regularization and early stopping, and for training one model.

In [14]:
def train_one_epoch(model, loader, optimizer, criterion, device, l1_lambda=0.0, l2_lambda=0.0, scaler=None):
    model.train()
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    total_correct = 0
    total_samples = 0

    for xb_dyn, xb_static, yb in loader:
        xb_dyn = xb_dyn.to(device)
        xb_static = xb_static.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                logits = model(xb_dyn, xb_static)
                loss = criterion(logits, yb)
                # L2 is handled by weight_decay in optimizer
                if l1_lambda > 0.0:
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + l1_lambda * l1_norm
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(xb_dyn, xb_static)
            loss = criterion(logits, yb)
            if l1_lambda > 0.0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        total_loss += loss.item() * yb.size(0)
        preds = logits.argmax(dim=1)
        all_predictions.append(preds.cpu().numpy())
        all_targets.append(yb.cpu().numpy())
        total_correct += (preds == yb).sum().item()
        total_samples += yb.size(0)
    
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average="weighted"
    )

    avg_loss = total_loss / total_samples
    return avg_loss, total_correct / total_samples, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for xb_dyn, xb_static, yb in loader:
            xb_dyn = xb_dyn.to(device)
            xb_static = xb_static.to(device)
            yb = yb.to(device)

            logits = model(xb_dyn, xb_static)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            preds = logits.argmax(dim=1)
            all_predictions.append(preds.cpu().numpy())
            all_targets.append(yb.cpu().numpy())
            total_correct += (preds == yb).sum().item()
            total_samples += yb.size(0)
    
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average="weighted"
    )

    avg_loss = total_loss / total_samples
    return avg_loss, total_correct / total_samples, epoch_f1


def train_model(model, train_loader, val_loader, criterion, learning_rate,
                l1_lambda, l2_lambda, epochs, patience, device, scaler=None, verbose=10):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=l2_lambda
    )

    best_val_loss = float("inf")
    best_val_f1 = 0.0
    best_state_dict = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            l1_lambda=l1_lambda, l2_lambda=l2_lambda, scaler=scaler
        )
        val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, device)

        if epoch % verbose == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} acc={train_acc:.4f} F1={train_f1:.4f} | "
                  f"val_loss={val_loss:.4f} acc={val_acc:.4f} F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                print(f"Best val F1 {best_val_f1}")
                break

    return best_state_dict, best_val_loss

## K-fold Ensemble Training

Train one model per fold. We save each fold's best weights and later ensemble their predictions.

In [15]:
PATIENCE = 40
K_FOLDS = 5

In [16]:
def train_kfold_ensemble(X_dyn, X_static, y, groups, params, n_splits=5, model_dir="models"):
    """Train an ensemble with K-fold CV without data leakage.

    Uses GroupKFold with groups=groups, so that all windows coming from the same
    original sample_index stay in the same fold.
    """
    os.makedirs(model_dir, exist_ok=True)
    gkf = GroupKFold(n_splits=n_splits)
    model_paths = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_dyn, y, groups=groups), 0):
        print(f"\n===== Fold {fold+1}/{n_splits} =====")
        X_tr_dyn, X_val_dyn = X_dyn[train_idx], X_dyn[val_idx]
        X_tr_stat, X_val_stat = X_static[train_idx], X_static[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        train_ds = TimeSeriesWindowDataset(X_tr_dyn, X_tr_stat, y_tr)
        val_ds   = TimeSeriesWindowDataset(X_val_dyn, X_val_stat, y_val)

        train_loader = make_loader(train_ds, batch_size=params["batch_size"], shuffle=True)
        val_loader   = make_loader(val_ds, batch_size=params["batch_size"] * 2, shuffle=False)

        model = BiGRUAttentionNetImproved(
            input_size=params["input_size"],
            hidden_size=params["hidden_size"],
            num_layers=params["num_layers"],
            num_classes=params["num_classes"],
            dropout=params["dropout_rate"],
            bidirectional=params["bidirectional"],
            batch_norm=params["batch_norm"],
            static_dim=params["static_dim"]
        ).to(DEVICE)

        scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

        best_state_dict, best_val_loss = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=params["criterion"],
            learning_rate=params["learning_rate"],
            l1_lambda=params["l1_lambda"],
            l2_lambda=params["l2_lambda"],
            epochs=params["epochs"],
            patience=params["patience"],
            device=DEVICE,
            scaler=scaler,
            verbose=10
        )

        # save best weights for this fold
        fold_path = os.path.join(model_dir, f"fold_{fold}_best.pt")
        torch.save(best_state_dict, fold_path)
        model_paths.append(fold_path)
        print(f"Saved best model for fold {fold} to {fold_path}")

    return model_paths

In [17]:
def group_logits_by_sample(logits_all_windows, window_ids):
    """
    Group window-level logits by sample_index.
    
    Args:
        logits_all_windows: Array (N_windows, num_classes)
        window_ids: Array (N_windows,) with sample_index for each window
    
    Returns:
        logit_sequences: List of arrays, each (num_windows_for_sample, num_classes)
        sample_ids: Array of unique sample IDs in order
    """
    unique_ids = np.unique(window_ids)
    logit_sequences = []
    
    for sid in unique_ids:
        mask = window_ids == sid
        sample_logits = logits_all_windows[mask]  # (num_windows_for_this_sample, num_classes)
        logit_sequences.append(sample_logits)
    
    return logit_sequences, unique_ids

## Prediction on Test Set

1. Each fold model predicts logits for each test window
2. Group logits by sample_index
3. Average all logits for windows from the same user and from all fold models

In [18]:
def predict_logits(model, loader, device):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for xb_dyn, xb_static, _ in loader:
            xb_dyn = xb_dyn.to(device)
            xb_static = xb_static.to(device)
            logits = model(xb_dyn, xb_static)
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)


def predict_label(model_paths, X_dyn_test, X_static_test, 
                      test_ids_win, base_params, label_encoder):
    # Collect logits from fold models
    print("Collecting logits from base ensemble...")
    test_logit_sequences, test_sample_ids = collect_logits_per_sample(
        model_paths, X_dyn_test, X_static_test, test_ids_win, base_params, DEVICE
    )
    
    print(f"Collected logits for {len(test_logit_sequences)} test samples")
    
    # Average logits across all windows
    mean_fold_logits = np.mean(test_logit_sequences, axis=1)  # (num_samples, num_classes)
    all_preds = mean_fold_logits.argmax(axis=1)
    preds_labels = label_encoder.inverse_transform(all_preds)
    
    print(f"Generated predictions for {len(preds_labels)} samples")
    
    return test_sample_ids, preds_labels

## Complete Example Workflow

In [19]:
# COMPLETE WORKFLOW WITH BiGRU-ATTENTION
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)

# Step 1: Set base model hyperparameters
best_params = {
    "input_size": X_dyn_win.shape[2],
    "static_dim": X_static_win.shape[1],
    "hidden_size": 256,
    "num_layers": 3,
    "num_classes": len(label_encoder.classes_),
    "dropout_rate": 0.3,
    "bidirectional": True,
    "batch_norm": True,
    # Training parameters
    "learning_rate": 1e-3,
    "l1_lambda": 1e-7, 
    "l2_lambda": 1e-5, 
    "epochs": 500,
    "patience": PATIENCE,
    "criterion": criterion,
    "batch_size": 64,
}

# Step 2: Train k-fold models
model_paths = train_kfold_ensemble(
    X_dyn_win, X_static_win, y_win, ids_win, 
    best_params, 
    n_splits=K_FOLDS, 
    model_dir="models"
)


===== Fold 1/5 =====
Epoch 001 | train_loss=0.7166 acc=0.8952 F1=0.8946 | val_loss=0.7620 acc=0.8602 F1=0.8621
Epoch 010 | train_loss=0.5239 acc=0.9919 F1=0.9919 | val_loss=0.6665 acc=0.9228 F1=0.9204
Epoch 020 | train_loss=0.5068 acc=0.9989 F1=0.9989 | val_loss=0.7282 acc=0.9028 F1=0.9012
Epoch 030 | train_loss=0.5001 acc=1.0000 F1=1.0000 | val_loss=0.6964 acc=0.9088 F1=0.9076
Epoch 040 | train_loss=0.5094 acc=0.9976 F1=0.9976 | val_loss=0.7160 acc=0.8972 F1=0.8964
Epoch 050 | train_loss=0.5023 acc=1.0000 F1=1.0000 | val_loss=0.7409 acc=0.9028 F1=0.9023
Early stopping at epoch 50
Best val F1 0.9203575940444277
Saved best model for fold 0 to models/fold_0_best.pt

===== Fold 2/5 =====
Epoch 001 | train_loss=0.7373 acc=0.8804 F1=0.8803 | val_loss=0.6514 acc=0.9348 F1=0.9341
Epoch 010 | train_loss=0.5306 acc=0.9869 F1=0.9869 | val_loss=0.6175 acc=0.9455 F1=0.9465
Epoch 020 | train_loss=0.5083 acc=0.9968 F1=0.9969 | val_loss=0.6161 acc=0.9505 F1=0.9504
Epoch 030 | train_loss=0.4988 acc=1

In [20]:
# Step 3: Make predictions on test set
sample_ids_ordered, preds_labels = predict_label(
    model_paths=model_paths,
    X_dyn_test=X_dyn_test_win,
    X_static_test=X_static_test_win,
    test_ids_win=test_ids_win,
    base_params=best_params,
    label_encoder=label_encoder
)

submission = pd.DataFrame({
    ID_COL: sample_ids_ordered,
    TARGET_COL: preds_labels
})
submission = submission.sort_values(ID_COL).reset_index(drop=True)
submission.to_csv("submission_bigru_attention_window.csv", index=False)

print("\n" + "="*60)
print("SUBMISSION FILE CREATED!")
print("="*60)
display(submission.head(10))
print(f"\nTotal samples: {len(submission)}")
print(f"Class distribution:\n{submission[TARGET_COL].value_counts()}")

Loading model from models/fold_0_best.pt
Loading model from models/fold_1_best.pt
Loading model from models/fold_2_best.pt
Loading model from models/fold_3_best.pt
Loading model from models/fold_4_best.pt
Collected logits for 1324 test samples
Generated predictions for 1324 samples

SUBMISSION FILE CREATED!


,sample_index,label
0,0,no_pain
1,1,no_pain
2,2,no_pain
3,3,no_pain
4,4,no_pain
5,5,no_pain
6,6,no_pain
7,7,no_pain
8,8,no_pain
9,9,no_pain



Total samples: 1324
Class distribution:
label
no_pain      1023
low_pain      181
high_pain     120
Name: count, dtype: int64
